# OASIS · training meja + orang (YOLO11s)

Jalankan di **Colab/Kaggle dengan GPU**. Hasil yang dibawa pulang: `best.pt` (dan `best.onnx`).

Dataset yang diharapkan: zip berisi
```
dataset/
  images/{train,val,test}/*.jpg
  labels/{train,val,test}/*.txt
```
Kelas: `0 person`, `1 table`, `2 chair`. Split **per sesi video**, bukan acak per frame.


In [ ]:
!nvidia-smi
!pip install -q ultralytics


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# upload dataset.zip ke Drive, atau pakai files.upload()
!cp /content/drive/MyDrive/oasis/dataset.zip /content/ && cd /content && unzip -q -o dataset.zip && ls dataset


In [ ]:
from pathlib import Path
root = Path('/content/dataset')
for split in ('train','val','test'):
    n = len(list((root/'images'/split).glob('*')))
    print(split, n)
assert (root/'images/train').exists(), 'taruh dataset di /content/dataset'


In [ ]:
Path('/content/oasis-tables.yaml').write_text('''path: /content/dataset
train: images/train
val: images/val
test: images/test
names:
  0: person
  1: table
  2: chair
''')
print(open('/content/oasis-tables.yaml').read())


## Train

`yolo11s` @ 960, 100 epoch, patience 30. Turunkan `imgsz` ke 640 kalau OOM di GPU kecil.


In [ ]:
!yolo detect train model=yolo11s.pt data=/content/oasis-tables.yaml \
  epochs=100 imgsz=960 batch=16 patience=30 close_mosaic=10 workers=8 \
  project=/content/runs name=table_counter_v1 exist_ok=True


## Metrik per kelas

Target minimum: `person` mAP50 ≥ 0.85 · `table` mAP50 ≥ 0.60 (prioritas orang, meja dikunci kalibrasi).


In [ ]:
!yolo detect val model=/content/runs/table_counter_v1/weights/best.pt \
  data=/content/oasis-tables.yaml imgsz=960
import pandas as pd
print(pd.read_csv('/content/runs/table_counter_v1/results.csv').tail(3)[['epoch','metrics/mAP50(B)','metrics/mAP50-95(B)']])


## MAE per meja (metrik yang paling penting)

Salin `cv/` dari repo ke Colab, lalu jalankan pipeline pada klip test yang tidak dipakai training:
```
python cv/calibrate.py --source test_clip.mp4 --zone-model runs/.../best.pt \
  --camera-id cam-test --space Library --labels L1,L2,L3,L4 --frames 40
python cv/count.py --source test_clip.mp4 --tables cv/tables/cam-test.json \
  --out /content/test.json --smooth 10
```
Bandingkan `samples[].zones[].people` dengan hitungan manual di 5 titik waktu; laporkan MAE.


In [ ]:
!cp /content/runs/table_counter_v1/weights/best.pt /content/drive/MyDrive/oasis/best.pt
!yolo export model=/content/runs/table_counter_v1/weights/best.pt format=onnx imgsz=640 half=False
!cp /content/runs/table_counter_v1/weights/best.onnx /content/drive/MyDrive/oasis/best.onnx
!ls -lh /content/drive/MyDrive/oasis/


## Di laptop

1. Taruh `best.pt` di `cv/runs/tables/weights/best.pt` (timpa yang lama).
2. Kalibrasi ulang tiap kamera: `python cv/calibrate.py --source <video> --zone-model best.pt ...`
3. Regenerasi replay: `python cv/count.py --source <video> --tables cv/tables/<cam>.json --out public/cam/<cam>.json --annotate ...` lalu `node scripts/build-cam.mjs`.
4. `npm run build && npm run test:ui` — tes UI memverifikasi badge `p/capacity · status` dan baris kamera.
